<a href="https://colab.research.google.com/github/deekshdechamma/DL_skill_developer/blob/main/dl3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 3: Vectorized Convolutional Forward Pass via im2col
Transformation from Scratch
● Objective: Understand the mechanics of spatial feature extractors by implementing
vectorized convolutional layers optimized for high-performance matrix multiplications.
● Required Tech Stack: Pure NumPy, Python 3.10+, SciPy.
● Task Description: Students must implement a 2D convolutional layer supporting arbitrary
padding, strides, and input/output channels from scratch. They are required to write an
im2col matrix helper to reshape the input tensor into a unified 2D matrix, transforming
sliding window convolutions into a single dot-product operation (
).

In [ ]:
#Step 1 — Import libraries
import numpy as np
from scipy import signal

In [1]:
#Step 2 — Create the im2col function
def im2col(x, kernel_h, kernel_w, stride=1, padding=0):

    # Input dimensions
    N, C, H, W = x.shape

    # Add zero padding
    x_padded = np.pad(
        x,
        ((0, 0), (0, 0),
         (padding, padding),
         (padding, padding)),
        mode='constant'
    )

    # Calculate output height and width
    out_h = (H + 2 * padding - kernel_h) // stride + 1
    out_w = (W + 2 * padding - kernel_w) // stride + 1

    # Store patches
    cols = []

    for i in range(out_h):
        for j in range(out_w):

            h_start = i * stride
            w_start = j * stride

            patch = x_padded[
                :,
                :,
                h_start:h_start + kernel_h,
                w_start:w_start + kernel_w
            ]

            cols.append(patch.reshape(N, -1))

    # Convert list of patches into matrix
    cols = np.stack(cols, axis=2)

    # Rearrange into im2col format
    cols = cols.transpose(1, 0, 2).reshape(
        C * kernel_h * kernel_w,
        -1
    )

    return cols, out_h, out_w

In [2]:
#Step 3 — Implement the vectorized convolution forward pass
def conv2d_forward(x, weights, bias, stride=1, padding=0):

    N, C, H, W = x.shape

    # Weight dimensions
    out_channels, in_channels, kernel_h, kernel_w = weights.shape

    # Check channel compatibility
    assert C == in_channels, "Input channels do not match filter channels"

    # Convert input into column matrix
    x_col, out_h, out_w = im2col(
        x,
        kernel_h,
        kernel_w,
        stride,
        padding
    )

    # Flatten filters
    w_col = weights.reshape(out_channels, -1)

    # Matrix multiplication
    out = w_col @ x_col

    # Add bias
    out = out + bias.reshape(-1, 1)

    # Reshape back to image format
    out = out.reshape(
        out_channels,
        N,
        out_h,
        out_w
    )

    out = out.transpose(1, 0, 2, 3)

    return out

In [4]:
# Step 4 — Create sample input

import numpy as np

np.random.seed(42)

x = np.random.randn(1, 1, 5, 5)

print(x)
print("Shape:", x.shape)

[[[[ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337]
   [-0.23413696  1.57921282  0.76743473 -0.46947439  0.54256004]
   [-0.46341769 -0.46572975  0.24196227 -1.91328024 -1.72491783]
   [-0.56228753 -1.01283112  0.31424733 -0.90802408 -1.4123037 ]
   [ 1.46564877 -0.2257763   0.0675282  -1.42474819 -0.54438272]]]]
Shape: (1, 1, 5, 5)


In [5]:
#Step 5 — Create convolution filters
weights = np.random.randn(2, 1, 3, 3)

bias = np.random.randn(2)

print("Weights shape:", weights.shape)
print("Bias shape:", bias.shape)

Weights shape: (2, 1, 3, 3)
Bias shape: (2,)


In [6]:
#Step 6 — Test im2col
x_col, out_h, out_w = im2col(
    x,
    kernel_h=3,
    kernel_w=3,
    stride=1,
    padding=1
)

print("im2col shape:", x_col.shape)
print("Output height:", out_h)
print("Output width:", out_w)

im2col shape: (9, 25)
Output height: 5
Output width: 5


In [7]:
#Step 7 — Run convolution
output = conv2d_forward(
    x,
    weights,
    bias,
    stride=1,
    padding=1
)

print("Output shape:", output.shape)
print("\nConvolution Output:")
print(output)

Output shape: (1, 2, 5, 5)

Convolution Output:
[[[[-2.02998802 -2.21556295  2.08795076 -0.13952963 -2.0245135 ]
   [-1.30782832 -1.73333416 -0.22186222 -0.42222931 -3.2595788 ]
   [ 2.05597415 -2.94803347 -0.74509603  4.06701568 -0.98817525]
   [ 1.04978016  3.36494424  0.80082917  2.92587949 -0.20228812]
   [-0.3261025   0.06525014 -0.14309622  0.95087238  2.23828033]]

  [[-2.38822658 -2.13180023 -0.41599115 -4.39321485 -4.40587241]
   [-1.51750171 -2.70346603 -6.35251874 -3.57732777 -1.44899235]
   [-0.3182426  -2.48957819 -1.58894587  1.80644497  2.6007737 ]
   [-0.18532382  2.30423804 -1.15832348  0.87688989  1.56338493]
   [-2.99470123 -3.19790695 -2.81264175  1.24636696  3.01386743]]]]


In [8]:
#Step 8 — Test arbitrary stride and padding
x2 = np.random.randn(2, 3, 8, 8)

weights2 = np.random.randn(4, 3, 3, 3)

bias2 = np.random.randn(4)

output2 = conv2d_forward(
    x2,
    weights2,
    bias2,
    stride=2,
    padding=1
)

print("Input shape :", x2.shape)
print("Weight shape:", weights2.shape)
print("Output shape:", output2.shape)

Input shape : (2, 3, 8, 8)
Weight shape: (4, 3, 3, 3)
Output shape: (2, 4, 4, 4)


In [9]:
#Step 9 — Show the matrix transformation
x_col, out_h, out_w = im2col(
    x,
    3,
    3,
    stride=1,
    padding=1
)

w_col = weights.reshape(2, -1)

print("Original Input Shape :", x.shape)
print("im2col Shape         :", x_col.shape)
print("Original Weight Shape:", weights.shape)
print("Weight Matrix Shape  :", w_col.shape)

result = w_col @ x_col

print("Matrix Result Shape  :", result.shape)

Original Input Shape : (1, 1, 5, 5)
im2col Shape         : (9, 25)
Original Weight Shape: (2, 1, 3, 3)
Weight Matrix Shape  : (2, 9)
Matrix Result Shape  : (2, 25)
